# Create an interface with fast strain matching

This notebook finds small periodic 2D supercells for a film/substrate interface under explicit constraints (area/strain/shear), using a deterministic matcher implemented in `utils/interface_matching.py`.

<h2 style="color:green">Usage</h2>

1. Select Input Materials (in the outer runtime) before running the notebook.
2. Set notebook parameters in cell 1.1. below (or use the default values).


## 1. Prepare the Environment
### 1.1. Set up the notebook 

Set the following flags to control the notebook behavior 

In [ ]:
FILM_INDEX = 1  # Index in the list of materials, to access as materials[FILM_INDEX]
FILM_MILLER_INDICES = (0, 0, 1)
FILM_THICKNESS = 1  # in atomic layers
FILM_TERMINATION_FORMULA = None  # if None, the first termination will be used
FILM_VACUUM = 0.0  # in angstroms
FILM_XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
FILM_USE_ORTHOGONAL_C = True

SUBSTRATE_INDEX = 0
SUBSTRATE_MILLER_INDICES = (1, 1, 1)
SUBSTRATE_THICKNESS = 3  # in atomic layers
SUBSTRATE_TERMINATION_FORMULA = None  # if None, the first termination will be used
SUBSTRATE_VACUUM = 0.0  # in angstroms
SUBSTRATE_XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
SUBSTRATE_USE_ORTHOGONAL_C = True

INTERFACE_DISTANCE = 3.0  # Gap between substrate and film, in Angstrom
INTERFACE_VACUUM = 10.0  # Vacuum over film, in Angstrom

# Whether to convert materials to conventional cells before creating slabs.
# To create interfaces with smaller cells, set this flag to False. (and pass already conventional cells as input)
USE_CONVENTIONAL_CELL = True

# Maximum area for the superlattice search algorithm (the final interface area will be smaller)


# Whether to reduce the resulting interface cell to the primitive cell after the interface creation.
# If the reduction causes unexpected results, try increasing the `MAX_AREA` for search.
REDUCE_RESULT_CELL_TO_PRIMITIVE = True

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install('mat3ra-utils')
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("create_interface_with_min_strain_zsl.ipynb")

### 1.3. Get input materials and assign `substrate` and `film`
Materials are loaded with `get_data()`. The first material is assigned as substrate and the second as film.

In [ ]:
from utils.jupyterlite import get_materials, set_materials, load_materials_from_folder

materials = get_materials(globals())
substrate = materials[SUBSTRATE_INDEX]
try:
    film = materials[FILM_INDEX]
except IndexError:
    print("Film material not found. Re-using substrate material as film.")
    film = substrate

### 1.4. Preview Substrate and Film

In [ ]:
from utils.visualize import visualize_materials as visualize

visualize([substrate, film], repetitions=[3, 3, 1], rotation="0x")

## 2. Configure slabs for interface


### 2.1. Get possible terminations for the slabs

In [ ]:
from mat3ra.made.tools.helpers import get_slab_terminations

film_slab_terminations = get_slab_terminations(material=film, miller_indices=FILM_MILLER_INDICES)
substrate_slab_terminations = get_slab_terminations(material=substrate, miller_indices=SUBSTRATE_MILLER_INDICES)
print("Film slab terminations:", film_slab_terminations)
print("Substrate slab terminations:", substrate_slab_terminations)

### 2.2. Select terminations
If termination formula is not provided, the first termination is used.


In [ ]:
from mat3ra.made.tools.helpers import select_slab_termination


film_termination = select_slab_termination(film_slab_terminations, FILM_TERMINATION_FORMULA)
substrate_termination = select_slab_termination(substrate_slab_terminations, SUBSTRATE_TERMINATION_FORMULA)

# Extract formula strings from termination objects
FILM_TERMINATION_FORMULA = film_termination.formula if hasattr(film_termination, 'formula') else str(film_termination)
SUBSTRATE_TERMINATION_FORMULA = substrate_termination.formula if hasattr(substrate_termination, 'formula') else str(substrate_termination)

print('Selected film termination:', FILM_TERMINATION_FORMULA)
print('Selected substrate termination:', SUBSTRATE_TERMINATION_FORMULA)

### 2.4. Create Substrate and Layer Slabs
Slab Configuration lets define the slab thickness, vacuum, and the Miller indices of the interfacial plane and get the slabs with possible terminations.
Define the substrate slab cell that will be used as a base for the interface and the film slab cell that will be placed on top of the substrate slab.

In [ ]:
from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabConfiguration, SlabBuilder

substrate_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=substrate,
    miller_indices=SUBSTRATE_MILLER_INDICES,
    number_of_layers=SUBSTRATE_THICKNESS,
    vacuum=0.0,
    termination_top_formula=SUBSTRATE_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

film_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=film,
    miller_indices=FILM_MILLER_INDICES,
    number_of_layers=FILM_THICKNESS,
    vacuum=0.0,
    termination_bottom_formula=FILM_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

substrate_slab = SlabBuilder().get_material(substrate_slab_config)
film_slab = SlabBuilder().get_material(film_slab_config)
visualize([substrate_slab, film_slab], rotation="0x")

## 3. Find coherent matches (fast matcher)
This uses a deterministic 2D supercell matcher (`utils/interface_matching.py`) designed for interactive twist exploration.


In [ ]:
import numpy as np
from types import SimpleNamespace

import importlib
import utils.interface_matching as interface_matching

interface_matching = importlib.reload(interface_matching)
FastCoherentMatcher2D = interface_matching.FastCoherentMatcher2D
rotation_matrix_2d = interface_matching.rotation_matrix_2d

# User controls for interactive matching
AUTO_TWIST = False
TWIST_ANGLE = 30.0  # used only if AUTO_TWIST=False
TWIST_MIN_DEG = 0.0
TWIST_MAX_DEG = 60.0
TWIST_STEP_DEG = 2.0

STRAIN_SHARE_TO_FILM = 1.0  # 1.0 => film takes all stretch; 0.0 => substrate takes all stretch

MAX_AREA = 750  # in Angstrom^2
# Additional fine-tuning parameters (increase values to get more strained matches):
MAX_AREA_TOLERANCE = 0.09  # in Angstrom^2
MAX_LENGTH_TOLERANCE = 0.05
MAX_ANGLE_TOLERANCE = 0.02

MAX_ABS_PRINCIPAL_STRAIN = 0.1
MAX_SHEAR_STRAIN = 0.1
MAX_EXTRA_ROTATION_DEG = 2.0     # additional rotation allowed by polar decomposition

# Matching pre-alignment: define twist relative to substrate a-vector along +x
def align_first_vector_to_x_right_handed(vectors_2x2: np.ndarray):
    v = np.array(vectors_2x2, dtype=float)
    if np.linalg.det(v) < 0:
        v[1] *= -1
    a = v[0]
    phi = np.degrees(np.arctan2(a[1], a[0]))
    R = rotation_matrix_2d(-phi)
    return v @ R, R

film_vectors = np.array(film_slab.lattice.vector_arrays[0:2])[:, :2]
substrate_vectors = np.array(substrate_slab.lattice.vector_arrays[0:2])[:, :2]

substrate_vectors_aligned, R_align = align_first_vector_to_x_right_handed(substrate_vectors)
canonicalize_obtuse_angle=True
film_vectors_aligned = film_vectors @ R_align

matcher = FastCoherentMatcher2D(
    film_vectors_aligned,
    substrate_vectors_aligned,
    max_area=MAX_AREA,
    max_results=50,
    max_abs_principal_strain=MAX_ABS_PRINCIPAL_STRAIN,
    max_shear_strain=MAX_SHEAR_STRAIN,
    max_extra_rotation_deg=MAX_EXTRA_ROTATION_DEG,
    max_length_rel_tol=0.15,
    max_angle_deg_tol=10.0,
    length_bin=0.5,
    angle_bin_deg=2.0,
)

if AUTO_TWIST:
    twist_values = np.arange(TWIST_MIN_DEG, TWIST_MAX_DEG + 1e-9, TWIST_STEP_DEG).tolist()
    per_twist_max_results = 3
    max_total_results = 50

    if hasattr(matcher, 'find_matches_over_twist'):
        results = matcher.find_matches_over_twist(
            twist_values,
            strain_share_to_film=STRAIN_SHARE_TO_FILM,
            allow_rotation_relaxation=True,
            per_twist_max_results=per_twist_max_results,
            max_total_results=max_total_results,
        )
    else:
        # Backward-compatible fallback if the class was imported from an older build.
        results = []
        for t in twist_values:
            results.extend(
                matcher.find_matches(
                    twist_deg=float(t),
                    strain_share_to_film=STRAIN_SHARE_TO_FILM,
                    allow_rotation_relaxation=True,
                )[:per_twist_max_results]
            )
        results.sort(key=lambda r: (r.max_abs_principal_strain, r.shear_strain, r.shape_penalty, r.match_area))
        results = results[:max_total_results]
else:
    results = matcher.find_matches(
        twist_deg=TWIST_ANGLE,
        strain_share_to_film=STRAIN_SHARE_TO_FILM,
        allow_rotation_relaxation=True,
    )

# Adapt results to the existing plotting helper (expects .match_area and .total_strain_percentage)
matches = [
    SimpleNamespace(
        match_area=r.match_area,
        total_strain_percentage=100.0 * r.von_mises_strain,
        twist_deg=r.twist_deg,
        film_supercell_matrix=r.film_supercell_matrix,
        substrate_supercell_matrix=r.substrate_supercell_matrix,
        deformation=r.film_to_substrate_deformation,
        film_deformation=getattr(r, 'film_deformation_applied', getattr(r, 'film_to_substrate_deformation', None)),
        substrate_deformation=getattr(r, 'substrate_deformation_applied', np.eye(2)),
        extra_rotation_deg=r.extra_rotation_deg,
        max_abs_principal_strain=r.max_abs_principal_strain,
        shear_strain=r.shear_strain,
    )
    for r in results
]

if AUTO_TWIST:
    print(f'Found {len(matches)} matches (best over twist sweep {TWIST_MIN_DEG}..{TWIST_MAX_DEG} step {TWIST_STEP_DEG})')
else:
    print(f'Found {len(matches)} matches at TWIST_DEG={TWIST_ANGLE}')


### 3.1. Plot matches by strain and area


In [ ]:
from utils.plot import plot_strain_vs_area

PLOT_SETTINGS = {
    'HEIGHT': 600,
    'X_SCALE': 'log',
    'Y_SCALE': 'log',
}

plot_strain_vs_area(matches, PLOT_SETTINGS)


### 3.2. Inspect the best match


In [ ]:
best = matches[0] if matches else None
if best is None:
    print('No matches found. Relax constraints or increase MAX_AREA.')
else:
    print('Area (Å²):', best.match_area)
    print('von Mises strain (%):', best.total_strain_percentage)
    print('max |principal strain| (%):', 100.0 * best.max_abs_principal_strain)
    print('shear strain (%):', 100.0 * best.shear_strain)
    print('extra rotation (deg):', best.extra_rotation_deg)
    print('Film supercell matrix (2x2):')
    print(best.film_supercell_matrix)
    print('Substrate supercell matrix (2x2):')
    print(best.substrate_supercell_matrix)
    print('Film->Substrate deformation F (2x2), where B_film @ F = B_substrate:')
    print(best.deformation)


## 4. Build the interface from a selected match

This constructs a periodic interface by (1) rotating into the same reference frame used for matching,
(2) applying the integer supercells, (3) straining the film to match the substrate in-plane, and (4) stacking.


In [ ]:
import numpy as np

from utils.fast_interface_builder import build_interface_from_2d_match

if best is None:
    raise ValueError('No match selected (best is None).')

interface = build_interface_from_2d_match(
    film_slab=film_slab,
    substrate_slab=substrate_slab,
    film_supercell_matrix_2x2=best.film_supercell_matrix,
    substrate_supercell_matrix_2x2=best.substrate_supercell_matrix,
    film_deformation_2x2=getattr(best, 'film_deformation', best.deformation),
    substrate_deformation_2x2=getattr(best, 'substrate_deformation', np.eye(2)),
    align_rotation_2x2=R_align,
    twist_deg=float(getattr(best, 'twist_deg', 0.0)),
    gap=INTERFACE_DISTANCE,
    vacuum=INTERFACE_VACUUM,
    xy_shift=[0.0, 0.0],
    reduce_result_cell_to_primitive=False,
)

print('Interface created')
print('atoms:', len(interface.basis.elements.ids))


### 4.1. Preview the interface


In [ ]:
from utils.visualize import visualize_materials as visualize

visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")


In [ ]:
interface.name = f"{interface.name}-{TWIST_ANGLE}"
set_materials(interface)

In [ ]:

from utils.jupyterlite import load_material_from_folder

interface_30 = load_material_from_folder("uploads", "30")
interface_45 = load_material_from_folder("uploads", "45")

visualize([interface_30, interface_45], viewer="wave")

o### 3.3. (Optional) Sweep twist angle
For interactive use, vary `TWIST_DEG` (and re-run section 3) or run a coarse sweep below.


In [ ]:
twists = list(range(0, 61, 5))
summary = []
for t in twists:
    r = matcher.find_matches(twist_deg=float(t), strain_share_to_film=STRAIN_SHARE_TO_FILM, allow_rotation_relaxation=True)
    if not r:
        summary.append((t, None, None))
        continue
    summary.append((t, 100.0 * r[0].von_mises_strain, r[0].match_area))

for row in summary:
    print(row)
